In [ ]:
!pip install pandas openpyxl requests

In [ ]:
import re
import html
import time
import random
from getpass import getpass

import requests
import pandas as pd
from openpyxl import load_workbook
from openpyxl.styles import Font, Alignment


In [ ]:
PRODUCT = input("Masukkan nama product: ").strip()
if not PRODUCT:
    raise ValueError("Nama product jangan dikosongin!!")

safe_name = re.sub(r"[^a-zA-Z0-9_-]+", "_", PRODUCT).strip("_")

OUTPUT_XLSX = f"/content/data1688_{safe_name}_FINAL.xlsx"
OUTPUT_CSV = f"/content/data1688_{safe_name}_FINAL.csv"
CHECKPOINT_CSV = f"/content/data1688_{safe_name}_checkpoint.csv"

print("\n--- AUTHORIZATION DATA1688 ---")

AUTHORIZATION = getpass("Paste Auth di sini: ").strip()
if not AUTHORIZATION:
    raise ValueError("Auth token harus diisi buat nembus API-nya.")

In [ ]:
# Endpoint API
API_URL = "https://dd.data1688.com/stage-api/es/search/searchGlobal"

# Parameter dasar pencarian
BASE_PARAMS = {
    "preciseMatching[0]": "1",
    "GOODS_DESC": PRODUCT,
    "datatype": "2",
    "DEST_COUNTRY_TC": "INDONESIA",
    "date[0]": "2026-08-01",
    "date[1]": "2026-09-01",
    "dataSource": "全球",
    "isInputPrecise": "false",
    "destState[0]": "全部",
    "relationFlag": "0",
    "csRelationFlag": "0",
    "matchingMethod": "1",
    "pageNo": "1",
    "pageSize": "20",
    "tabName": "数据列表",
    "isExportFlag": "false",
    "sort": "query_date",
    "order": "desc",
}

HEADERS = {
    "Accept": "application/json, text/plain, */*",
    "Authorization": AUTHORIZATION,
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36",
    "Referer": "https://dd.data1688.com/",
    "Accept-Language": "en-US,en;q=0.9,id;q=0.8",
}

session = requests.Session()
session.headers.update(HEADERS)

TRANSLATION_MAP = {
    "进口": "Import", "出口": "Export",
    "印度尼西亚": "Indonesia", "澳大利亚": "Australia",
    "巴西": "Brazil", "日本": "Japan", "新西兰": "New Zealand",
    "美国": "United States", "加拿大": "Canada", "印度": "India",
    "中国": "China", "阿根廷": "Argentina", "乌拉圭": "Uruguay",
    "巴拉圭": "Paraguay", "智利": "Chile", "墨西哥": "Mexico",
    "西班牙": "Spain", "法国": "France", "德国": "Germany",
    "英国": "United Kingdom", "意大利": "Italy", "荷兰": "Netherlands",
    "爱尔兰": "Ireland", "丹麦": "Denmark", "比利时": "Belgium",
    "南非": "South Africa",
}


def clean_html(val):
    if not val: return None
    val = html.unescape(str(val))
    val = re.sub(r"<[^>]*>", "", val)
    return re.sub(r"\s+", " ", val).strip()

def translate_value(val):
    val = clean_html(val)
    if not val: return None
    return TRANSLATION_MAP.get(val, val)

def random_delay():
    jeda = random.uniform(2.0, 4.0)
    print(f"   [+] loading {jeda:.2f} detik...")
    time.sleep(jeda)

def fetch_page(page_number, max_retries=5):
    params = BASE_PARAMS.copy()
    params["pageNo"] = str(page_number)

    for attempt in range(1, max_retries + 1):
        try:
            print(f"-> Tarik halaman {page_number} (Coba {attempt}/{max_retries})")
            resp = session.get(API_URL, params=params, timeout=30)

            if resp.status_code == 429:
                wait_time = 20 * attempt
                print(f"   [!] Kena rate limit, nunggu {wait_time} detik dulu.")
                time.sleep(wait_time)
                continue

            if 500 <= resp.status_code <= 599:
                print("   [!] Server 1688 error, retrying...")
                time.sleep(5 * attempt)
                continue

            resp.raise_for_status()

            try:
                res_json = resp.json()
            except:
                raise ValueError("Response dari server bukan format JSON.")

            api_code = res_json.get("code")
            if api_code == 401:
                raise ValueError("Auth-nya expired atau salah nih. Ambil ulang dari Network ya.")
            if api_code != 200:
                raise ValueError(f"API Error {api_code}: {res_json.get('msg')}")

            return res_json.get("data", {})

        except requests.RequestException as e:
            print(f"   Jaringan ngelag: {e}")
            if attempt == max_retries:
                raise
            time.sleep(5 * attempt)

    raise Exception(f"Gagal narik data halaman {page_number} setelah retries.")

def save_checkpoint(data_list):
    if data_list:
        pd.json_normalize(data_list).to_csv(CHECKPOINT_CSV, index=False, encoding="utf-8-sig")

In [ ]:
print(f"\n=== MULAI SCRAPING: {PRODUCT.upper()} ===")

all_records = []

print("\nNarik metadata dari halaman 1...")
first_data = fetch_page(1)

total_rows = int(first_data.get("totalRow", 0))
total_pages = int(first_data.get("totalPage", 1))
records = first_data.get("param", []) or []

all_records.extend(records)
save_checkpoint(all_records)

print(f"\nInfo Target:")
print(f"- Total baris data API: {total_rows}")
print(f"- Total halaman     : {total_pages}")
print(f"- Dapet di Hal 1    : {len(records)} baris\n")

for page_no in range(2, total_pages + 1):
    random_delay()
    data = fetch_page(page_no)

    page_records = data.get("param", []) or []
    all_records.extend(page_records)

    save_checkpoint(all_records)
    print(f"   [v] Halaman {page_no} beres. Total dapet sejauh ini: {len(all_records)} baris")

print("\n=== PROSES TARIK DATA SELESAI ===")


=== MULAI SCRAPING: BEEF KNUCKLE ===

Narik metadata dari halaman 1...
-> Tarik halaman 1 (Coba 1/5)

Info Target:
- Total baris data API: 0
- Total halaman     : 0
- Dapet di Hal 1    : 0 baris


=== PROSES TARIK DATA SELESAI ===


In [ ]:
def format_record(r):
    # Mapping
    ds = translate_value(r.get("SOURCE"))
    pc = translate_value(r.get("DEST_COUNTRY_TC_KEYWORD"))

    return {
        "Data source": ds if ds else "Indonesia",
        "Import & Export": translate_value(r.get("EXP_IMP")),
        "date": clean_html(r.get("DATE")),
        "HS code": clean_html(r.get("HS_CODE")),
        "Product DESC": clean_html(r.get("GOODS_DESC")),
        "Buyers": clean_html(r.get("CONSIGNEE")),
        "Supplier": clean_html(r.get("SHIPPER")),
        "Weight": pd.to_numeric(r.get("WEIGHT"), errors="coerce"),
        "Quantity": pd.to_numeric(r.get("QUANTITY"), errors="coerce"),
        "Amount": pd.to_numeric(r.get("VALUE"), errors="coerce"),
        "Purpose Country": pc if pc else "Indonesia",
        "Origin Country": translate_value(r.get("ORIGIN_COUNTRY_TC_KEYWORD"))
    }

print("Mulai rapihin data mentah ke tabel...")
clean_rows = [format_record(r) for r in all_records]
df = pd.DataFrame(clean_rows)

df["HS code"] = df["HS code"].astype("string")

df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")

raw_df = pd.json_normalize(all_records)
with pd.ExcelWriter(OUTPUT_XLSX, engine="openpyxl") as writer:
    df.to_excel(writer, sheet_name="Data", index=False)
    raw_df.to_excel(writer, sheet_name="Raw API", index=False)

wb = load_workbook(OUTPUT_XLSX)
ws = wb["Data"]
ws.freeze_panes = "A2"
ws.auto_filter.ref = ws.dimensions

for cell in ws[1]:
    cell.font = Font(bold=True)
    cell.alignment = Alignment(horizontal="center", vertical="center")

col_widths = {"A":18, "B":18, "C":15, "D":15, "E":60, "F":40, "G":40, "H":16, "I":16, "J":18, "K":22, "L":22}
for col, width in col_widths.items():
    ws.column_dimensions[col].width = width

for row in ws.iter_rows(min_row=2):
    for i in [4, 5, 6]:
        row[i].alignment = Alignment(vertical="top", wrap_text=True)

for cell in ws["D"][1:]:
    cell.number_format = "@"

wb.save(OUTPUT_XLSX)
print("File Excel udah berhasil dibuat dan diformat!")

Mulai rapihin data mentah ke tabel...
File Excel udah berhasil dibuat dan diformat!


In [ ]:
print("\n--- VALIDASI AKHIR ---")
print(f"Target jumlah data dari web : {total_rows}")
print(f"Total yang berhasil discrap : {len(df)}")

if len(df) == total_rows:
    print("Status: MANTAP! COCOK :)")
else:
    print("Status: ADA YANG MISSING / LEBIH :(")

print("\nCek kolom yang datanya kosong (NaN):")
print(df.isna().sum())

print("\nPreview 5 data pertama:")
display(df.head())

print(f"- {OUTPUT_XLSX}")
print(f"- {OUTPUT_CSV}")

print("\nOtw download file Excel-nya...")
files.download(OUTPUT_XLSX)


--- VALIDASI AKHIR ---
Target jumlah data dari web : 250
Total yang berhasil discrap : 250
Status: MANTAP! COCOK :)

Cek kolom yang datanya kosong (NaN):
Data source        0
Import & Export    0
date               0
HS code            0
Product DESC       0
Buyers             0
Supplier           0
Weight             0
Quantity           0
Amount             0
Purpose Country    0
Origin Country     0
dtype: int64

Preview 5 data pertama:


,Data source,Import & Export,date,HS code,Product DESC,Buyers,Supplier,Weight,Quantity,Amount,Purpose Country,Origin Country
0,Indonesia,Import,2026-08-31,02023000,FROZEN BONELESS BEEF A KNUCKLE,INDOKULINA SARANA UTAMA,HARVEY INDUSTRIES GROUP PTY LTD,1617.14,1617.14,13745.69,Indonesia,Australia
1,Indonesia,Import,2026-08-29,02023000,1096 CARTONS FROZEN BONELESS BEEF KNUCKLES,BERDIKARI,SURE GOOD FOODS LTD,28014.22,28.01,171391.01,Indonesia,Brazil
2,Indonesia,Import,2026-08-29,02023000,1060 CARTONS FROZEN BONELESS BEEF KNUCKLES,BERDIKARI,SURE GOOD FOODS LTD,28142.81,28.14,172177.69,Indonesia,Brazil
3,Indonesia,Import,2026-08-29,02023000,1088 CARTONS FROZEN BONELESS BEEF KNUCKLES,BERDIKARI,SURE GOOD FOODS LTD,27992.05,27.99,171255.39,Indonesia,Brazil
4,Indonesia,Import,2026-08-29,02023000,1099 CARTONS FROZEN BONELESS BEEF KNUCKLES,BERDIKARI,SURE GOOD FOODS LTD,27713.51,28.01,169551.26,Indonesia,Brazil


- /content/data1688_beef_knuckle_FINAL.xlsx
- /content/data1688_beef_knuckle_FINAL.csv

Otw download file Excel-nya...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>